# 03 — XGB PP+HP Two-Stage Stage 2 회귀 (y>0 only) (스태킹 다양성용)

기존 `reg/xgb.ipynb`(HP만 흔드는 노트북)의 **쌍둥이** — 여기에 **PP 6축을 trial 축에 추가**해서 같은 모델이라도 다른 전처리에서 학습한 OOF를 만든다 → 스태킹 base 다양성 ↑.

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_*_data.csv`
- **출력**: `4_output/03_two_stage/default/reg/xgb/pphp/{best_params.json, fold_models.pkl, optuna_*.db, oof|val|test_die.csv, oof|val|test_unit.csv}`
- **PP**: 6축 Optuna 탐색 ([pp_hp_strategy.md §3](../../../../../pp_hp_strategy.md)) — `missing_threshold / corr_threshold / add_indicator / indicator_threshold / spatial_max_dist / post_impute_corr_threshold`. 나머지는 PP_FIXED와 동일하게 고정. trial마다 `pp_hpo.make_cached_preprocess`로 전처리 (LRU 캐시).
- **HP**: hp-only(`reg/xgb.ipynb`)와 **동일 범위** — `models.get_search_space('xgb')`. anchor도 동일 HP + PP_FIXED 값(`pp_*` 키, corr 0.90→0.88).
- **target transform**: `'none'` 고정 (strategy_common §24) · y>0 die만 학습 (Y_POSITIVE_ONLY)
- **CV / 후처리**: hp-only와 동일 — unit-level 5-fold KFold, postprocess 집계 8종 + zero_clip.

> ⚠ pp+hp는 trial마다 전처리(spatial impute 포함)를 다시 도므로 hp-only보다 느리다. `PP_CACHE_SIZE`로 캐시 보관 개수 조정 (메모리 vs 속도). `N_TRIALS`는 환경에 맞춰 줄여도 됨.

## 1. 환경 설정 + 모듈 import

In [ ]:
import os, sys

# Google Drive 파일 ID들 — Colab에서 코드/데이터/모듈 zip을 자동으로 받아 풀 때 사용 (로컬은 무시)
GDRIVE_CODE_ID          = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'  # code.zip = setup.py + utils/
GDRIVE_DATASET_ID       = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'  # dataset.zip = 원본 CSV 4개
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'  # preprocessing.zip = cleaning/outlier/scaling 등
GDRIVE_MODELING_ID      = '1Vrn5LBl611rWbag7d09LZH68_lfpu6wP'   # modeling.zip = 3_modeling/modules (코드 수정 시 재업로드)
GDRIVE_OUTPUT_ID       = '1ts73qEMmjX8cKIb-QeDQ-TMeyudFGWzs'  # 4_output.zip = 기존 실험 산출물 (RESUME용)
RESUME                 = True   # True=기존 optuna db에 trial 이어 붙임 / False=처음부터 (db 있으면 의도적 에러)

# Colab이면 필요한 zip들을 받아 풀고(이미 풀려 있으면 skip), 로컬이면 ../../../../..만
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if not os.path.exists('/content/project/3_modeling/modules/pp_hpo.py'):
        assert GDRIVE_MODELING_ID, 'GDRIVE_MODELING_ID가 비어있음 — modules.zip Drive ID 입력 필요'
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modules.zip')
        os.system('unzip -qo /content/modules.zip -d /content/project/3_modeling')
    if RESUME and GDRIVE_OUTPUT_ID and not os.path.exists('/content/project/4_output/01_zit'):
        os.system(f'gdown {GDRIVE_OUTPUT_ID} -O /content/4_output.zip')
        os.system('unzip -qo /content/4_output.zip -d /content/project')
        os.remove('/content/4_output.zip')
    # 모듈 경로 등록 — Colab 런타임은 세션마다 sys.path가 초기화됨
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# 공통 유틸: 경로 상수(OUTPUT_DIR, DATA_DIR), 컬럼 상수(TARGET_COL, KEY_COL), SEED
from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

# 전처리 모듈(2_preprocessing) 경로 + `from modules import ...` 가 3_modeling/modules를 찾게
# 전처리 모듈(2_preprocessing/)과 모델링 모듈(3_modeling/modules/) 모두 sys.path 등록
# 노트북 위치가 달라도 PROJECT_ROOT 기준 절대경로로 접근 → Colab·로컬 동일
PREP_ROOT = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PREP_ROOT not in sys.path:
    sys.path.insert(0, PREP_ROOT)
MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

# hpo: run_hpo/refit_best/save_artifacts
# pp_hpo: PP를 trial 6축에 추가한 래퍼 (run_pp_hpo/run_pp_clf_hpo/refit_pp_best/make_cached_preprocess)
# models: AVAILABLE_MODELS + CLF_AVAILABLE_MODELS 레지스트리
from modules import hpo, models, pp_hpo   # pp_hpo: PP를 trial 축에 넣는 래퍼 (run_pp_hpo / run_pp_clf_hpo / refit_pp_best / make_cached_preprocess)
# meta_features: position·die_xy 메타피처 생성 (make_cached_preprocess 내부에서도 사용)
from meta_features import add_meta_features   # (pp_hpo.make_cached_preprocess가 내부에서 사용 — import 가능 확인용)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'Available models: {models.AVAILABLE_MODELS} | clf: {models.CLF_AVAILABLE_MODELS}')

## 2. 실험 설정

In [ ]:
# 모델 고정 (이 노트북은 XGB Two-Stage Stage 2 회귀 (y>0 only), PP+HP 동시 탐색)
MODEL_NAME = 'xgb'
EXP_ID     = 'ts-reg-xgb-pphp'
EXP_MEMO   = 'PP+HP 동시 탐색 (스태킹 다양성). HP 범위·anchor·CV·후처리는 reg/xgb.ipynb와 동일, PP만 6축 추가'
USER       = 'jh'

# Optuna 예산
N_TRIALS = 3000
N_FOLDS  = 5
# N_STARTUP_TRIALS: 랜덤 탐색 후 TPE 전환 — 초반 공간 커버리지 확보
N_STARTUP_TRIALS = 50

N_JOBS = -1   # 모델 학습 병렬도 (-1 = 전체 코어). strategy_common §8: 노트북 여러 개 병렬이면 7 등으로 낮출 것
TIMEOUT_SEC = 20 * 60 * 60  # 초 단위, None=무제한 (Colab 타임아웃 대비)

TARGET_TRANSFORM = 'none'   # 트리는 'none' 통일
Y_POSITIVE_ONLY  = True   # Two-Stage Stage 2 — y>0 die만 학습
CLIP_Y_EXTREME   = True

# PP_CACHE_SIZE: trial마다 전처리 재실행 대신 캐싱 (같은 PP 조합 재사용 시 속도 향상)
PP_CACHE_SIZE = 2   # cached_preprocess가 보관할 PP 조합 개수 (cleaned 3-split ≈ 1~1.5GB/개) — 메모리 보고 조정

# 출력 경로 — EXP_ID 끝자리('pphp')를 하위 폴더명으로 (hp-only와 분리)
OUT_DIR = os.path.join(OUTPUT_DIR, '03_two_stage', 'default', 'reg', 'xgb', 'pphp', '001')
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')
os.makedirs(OUT_DIR, exist_ok=True)

# anchor — hp-only(reg/xgb.ipynb)의 1차 best HP + PP_FIXED 값(pp_* 키).
# ⚠ pp_* 값은 pp_hpo.PP_SEARCH_CANDIDATES 후보 안에 있어야 함. PP_FIXED corr_threshold=0.90은 후보 [0.80,0.84,0.88,0.92,0.96,0.98]에 없어 0.88로 둠.
XGB_ANCHOR_PPHP = {
    'n_estimators': 1423,
    'learning_rate': 0.0363,
    'max_depth': 10,
    'min_child_weight': 0.621,
    'subsample': 0.728,
    'colsample_bytree': 0.618,
    'reg_alpha': 0.0168,
    'reg_lambda': 3.89e-06,
    'gamma': 3.84e-06,
    # PP anchor: PP_FIXED 고정값을 PP_SEARCH_CANDIDATES 후보 중 가장 가까운 값으로 매핑
    'pp_missing_threshold': 0.3,
    'pp_corr_threshold': 0.88,
    'pp_add_indicator': True,
    'pp_indicator_threshold': 0.05,
    'pp_spatial_max_dist': 6.0,
    'pp_post_impute_corr_threshold': 0.96,
}

print(f'EXP: {EXP_ID} | USER: {USER}')
print(f'N_TRIALS={N_TRIALS}, N_FOLDS={N_FOLDS}, N_JOBS={N_JOBS}, TIMEOUT_SEC={TIMEOUT_SEC}, PP_CACHE_SIZE={PP_CACHE_SIZE}')
print(f'TARGET_TRANSFORM={TARGET_TRANSFORM} | CLIP_Y_EXTREME={CLIP_Y_EXTREME} | Y_POSITIVE_ONLY={Y_POSITIVE_ONLY}')
print(f'OUT_DIR={OUT_DIR}')

## 3. 데이터 로드 + target clip

In [ ]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
# split_xs: xs['split'] 컬럼 기준으로 train/val/test 행 분리
xs_dict = split_xs(xs)
print(f'xs: {xs.shape}, feat_cols: {len(feat_cols)}')

# train y의 극단값(1.0, 1건)만 두 번째로 큰 값으로 clip — 학습 입력 안정화 (원본 ys는 보존)
# ys는 원본 보존 — clip/transform은 ys_input 복사본에만 적용 (누수 방지)
ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개')

# 트리 모델은 target 변환이 결과를 거의 안 바꿔서 'none'으로 고정 (clf는 안 씀 — 둘 다 None)
# 트리는 target 변환이 RMSE를 유의하게 낮추지 않음 (EDA max|r|=0.037 약신호)
target_transform_fn = None
target_inverse_fn   = None
print(f'[target transform] none (strategy_common §24 — 트리 target_transform=none 통일)')

## 4. 전처리 캐시 준비 (PP는 Optuna 6축 — pp_hpo)

In [ ]:
# PP는 더 이상 고정값이 아니라 Optuna 탐색 6축 (pp_hpo.PP_SEARCH_CANDIDATES — pp_hp_strategy.md §3)
print('[PP 탐색 6축]')
for k, v in pp_hpo.PP_SEARCH_CANDIDATES.items():
    print(f'  {k:28s} {v}')
print('  (고정: corr_keep_by=std, post_impute_corr_keep_by=std, const_threshold=1e-6, remove_duplicates=True,\n'
      '         imputation_method=spatial, outlier=winsorize 0.0/0.99, Stage0 EXCLUDE_COLS)')

# (xs, ys, feat_cols, xs_dict) 고정 → trial마다 호출하는 캐시된 전처리 함수.
# preprocess.run(Stage0→cleaning→winsorize) + add_meta_features(position='raw', die_xy)를 PP 조합별로 캐시.
# PP 조합당 1개 캐시 엔트리 (LRU) — 같은 PP 조합이 여러 trial에서 재사용될 때 속도 향상
cached_prep = pp_hpo.make_cached_preprocess(
    xs, ys_input, feat_cols, xs_dict,
    position_mode='raw', use_die_xy=True,
    maxsize=PP_CACHE_SIZE, suppress_stdout=True,
)
# suppress_stdout=True: 전처리 진행 출력을 숨겨 HPO 로그 오염 방지
print(f'\n[cached_preprocess 준비 완료] maxsize={PP_CACHE_SIZE}')

## 5. Optuna PP+HP HPO (anchor 첫 trial enqueue + PP 6축 + HP wide range)

In [ ]:
# study_meta: HPO 실험 메타정보를 DB에 박제 → trial별 EXP_ID/anchor/PP 설정 조회 가능
study_meta_for_save = {
    'exp_id':               EXP_ID,
    'exp_memo':             EXP_MEMO,
    'user':                 USER,
    'model_name':           MODEL_NAME,
    'context':              "ts_reg (y_positive_only)",
    'target_transform':     TARGET_TRANSFORM,
    'y_positive_only':      Y_POSITIVE_ONLY,
    'clip_y_extreme':       CLIP_Y_EXTREME,
        # pp_search_candidates: PP 6축 탐색 공간 snapshot → 재현성 확보
    'pp_search_candidates': pp_hpo.PP_SEARCH_CANDIDATES,
    # pp_cache_size: LRU 캐시 크기 (메모리 사용량 조절 파라미터)
    'pp_cache_size':        PP_CACHE_SIZE,
    'n_trials':             N_TRIALS,
    'n_folds':              N_FOLDS,
    'n_jobs':               N_JOBS,
        # n_startup_trials: TPE 이전 랜덤 탐색 횟수 — 공간 초기 커버리지 확보
    'n_startup_trials':     N_STARTUP_TRIALS,
    'timeout_sec':          TIMEOUT_SEC,
    'seed_kfold':           SEED,
    'anchor':               XGB_ANCHOR_PPHP,
}

# sampler/pruner — hp-only 노트북과 동일 (seed=None 다양성, MedianPruner는 trial.report 없으면 no-op)
# Optuna study — storage=SQLite로 trial 결과 영구 저장 (Colab 재시작 후 RESUME 가능)
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

# HPO 본체: 매 trial PP 6축 + HP 샘플 → unit 단위 KFold OOF → unit RMSE minimize.
# anchor는 enqueue_trials로 trial 0에 강제 (RESUME 시 기존 trial 있으면 자동 skip). val/test RMSE는 매 trial user_attr.
res = pp_hpo.run_pp_hpo(
    cached_prep,
    xs_dict['train'], ys_input['train'],
    MODEL_NAME,
    n_trials=N_TRIALS,
    n_folds=N_FOLDS,
        # n_jobs: 병렬 fold 학습 (-1=전체 코어). N_FOLDS=5이므로 최대 5 worker가 동시 학습
    n_jobs=N_JOBS,
    y_positive_only=Y_POSITIVE_ONLY,
        # target_transform_fn=None: 트리는 변환 없음 → 예측도 역변환 없이 그대로
    target_transform_fn=target_transform_fn,
    target_inverse_fn=target_inverse_fn,
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    resume_study=RESUME,
        # TPESampler: multivariate=True → 파라미터 간 상관 모델링. seed=None → 실행마다 다른 탐색 경로
    sampler=TPESampler(seed=None, multivariate=True, group=True, n_startup_trials=N_STARTUP_TRIALS),
        # MedianPruner: n_warmup=10 trial 이후 중간값 미달 trial 조기 중단 → 탐색 효율 향상
    pruner=MedianPruner(n_warmup_steps=10),
        # enqueue_trials: trial 0에 anchor 강제 → RESUME 시 기존 trial 있으면 자동 skip
    enqueue_trials=[XGB_ANCHOR_PPHP],
    timeout=TIMEOUT_SEC,
    user_attrs=study_meta_for_save,
    xs_val_raw=xs_dict['validation'], ys_val_unit=ys_input['validation'],
    xs_test_raw=xs_dict['test'],       ys_test_unit=ys_input['test'],
)
# res에서 best trial 정보 추출 → 재학습(다음 cell)에 사용
study                 = res['study']
best_params_for_refit = res['best_params']
# best_value: HPO 목적함수(OOF RMSE) 최솟값 → study_meta에 저장하여 산출물과 함께 기록
study_meta_for_save['hpo_best_value'] = float(res['best_value'])

# trial 0이 anchor 키들을 그대로 갖고 있는지 확인 (enqueue 정상 동작 검증)
first_trial_params = study.trials[0].params
anchor_keys_present = {k: first_trial_params.get(k) for k in XGB_ANCHOR_PPHP if k in first_trial_params}
print(f'\n[HPO 완료] best OOF RMSE = {res["best_value"]:.6f}')
print(f'[검증] trial 0 params (anchor 키만): {anchor_keys_present}')
print(f'[PP 캐시] {cached_prep.counters}')
print(f'best_params = {best_params_for_refit}')

## 6. Best trial 재학습 (best PP로 전처리 재실행 + K-fold OOF)

In [ ]:
# best PP로 preprocess 재실행 → 그 데이터로 best HP 5-fold 재학습 (refit_pp_best가 hpo.refit_best 위임).
# refit_pp_best: best PP로 전처리 재실행 후 best HP 5-fold 재학습 → die·unit 레벨 OOF/val/test 예측
out = pp_hpo.refit_pp_best(
    cached_prep, xs_dict, ys_input,
    MODEL_NAME, best_params_for_refit,
    n_folds=N_FOLDS, n_jobs=N_JOBS,
    y_positive_only=Y_POSITIVE_ONLY,
    target_transform_fn=target_transform_fn,
    target_inverse_fn=target_inverse_fn,
)
final           = out['refit_result']
xs_train        = out['xs_train']
xs_val          = out['xs_val']
xs_test         = out['xs_test']
    # best HP + best PP 조합으로 최종 모델 확정 — 재현성을 위해 fold_models.pkl에 저장
feat_cols_clean = out['feat_cols']
# effective_pp_params: 실제 사용된 PP 파라미터 → save_artifacts가 산출물에 기록
study_meta_for_save['effective_pp_params'] = out['effective_pp_params']   # save_artifacts가 top-level effective_pp_params로 기록
study_meta_for_save['best_pp_params']      = out['pp_params']
print(f"[best PP] {out['pp_params']}")
print(f"[best PP 전처리 후 feat_cols] {len(feat_cols_clean)}")
print(f"[PP 캐시 최종] {cached_prep.counters}")

# 후처리 이전(mean 집계) unit RMSE — train(OOF) / val / test
# die-level 예측을 unit-level mean 집계 후 RMSE 계산 (후처리 최적화 이전 baseline)
y_true = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
oof_u  = final['oof_pred_unit'].set_index(KEY_COL)['pred'].loc[y_true.index]
oof_rmse = float(np.sqrt(np.mean((oof_u.values - y_true.values)**2)))

# val·test는 fold 평균 예측 (각 fold 모델이 동일 val/test에 predict → 평균)
y_val_true  = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
val_u       = final['val_pred_unit'].set_index(KEY_COL)['pred'].loc[y_val_true.index]
val_rmse    = float(np.sqrt(np.mean((val_u.values - y_val_true.values)**2)))

y_test_true = ys_input['test'].set_index(KEY_COL)[TARGET_COL]
test_u      = final['test_pred_unit'].set_index(KEY_COL)['pred'].loc[y_test_true.index]
test_rmse   = float(np.sqrt(np.mean((test_u.values - y_test_true.values)**2)))

# segment 분해: train y max=1.0 / val y max≈0.17 → RMSE 스케일 차이가 크므로 세트별 함께 확인
print(f'\n[Refit 완료] (original space, postprocess 이전)')
print(f'  OOF  unit RMSE = {oof_rmse:.6f}')
print(f'  val  unit RMSE = {val_rmse:.6f}')
print(f'  test unit RMSE = {test_rmse:.6f}')

## 7. 후처리 매트릭스 + 산출물 저장

In [ ]:
# 후처리 설정 — die→unit 집계 8종 중 best + zero_clip 임계값 탐색. trees는 log space 안 씀(target_transform='none'), π threshold 없음
# 후처리: die→unit 집계 방식 8종 × zero_clip 임계값 그리드 탐색 → val RMSE 최소 조합 선택
POSTPROCESS_CONFIG = {
    'agg_methods':         ('mean', 'median', 'max', 'min', 'trimmed_mean', 'weighted', 'Q25', 'Q75'),
    'zero_clip_range':     (0.001, 0.015),
    'zero_clip_step':      0.001,
    'zero_clip_log_space': TARGET_TRANSFORM == 'log1p',
    'use_pi_threshold':    False,
}

# save_artifacts: fold_models.pkl + best_params.json + die·unit CSV 6개 → stacking 재사용 가능
hpo.save_artifacts(
    refit_result=final,
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    out_dir=OUT_DIR, exp_id=EXP_ID,
    feature_names=feat_cols_clean,
    extra_feature_name=None,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],
    y_test_unit=ys_input['test'],
    postprocess_config=POSTPROCESS_CONFIG,
    study_meta=study_meta_for_save,
)

# 저장 완료 후 파일 목록 출력 (파일명 + 크기) — 누락 파일 즉시 확인
for f in sorted(os.listdir(OUT_DIR)):
    size_kb = os.path.getsize(os.path.join(OUT_DIR, f)) / 1024
    print(f'  {f:30s}  {size_kb:10,.1f} KB')

# Colab이면 산출물을 zip으로 묶어 로컬 PC로 다운로드
# Colab 환경이면 OUT_DIR을 zip으로 묶어 로컬 PC에 자동 다운로드
try:
    import google.colab
    from google.colab import files
    import shutil
    _zip_path = shutil.make_archive(os.path.join('/content', f'{MODEL_NAME}_{EXP_ID}_outputs'), 'zip', OUT_DIR)
    print(f'[zip 생성] {_zip_path} ({os.path.getsize(_zip_path)/1024:.1f} KB)')
    try:
        files.download(_zip_path)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크 클릭')
        display(FileLink(_zip_path))
except ImportError:
    pass